<a href="https://colab.research.google.com/github/praju-007anchan/IN126060102_NLP/blob/main/NLP_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:

def clean_response(response):
    bad_words = ["autism", "hate", "stupid", "idiot"]

    for word in bad_words:
        if word in response.lower():
            return "I'm sorry, I couldn't understand that properly. Could you rephrase?"

    return response

In [4]:
def rule_based_response(user_input):
    user_input = user_input.lower()

    if "what is ai" in user_input or "artificial intelligence" in user_input:
        return "Artificial Intelligence is the simulation of human intelligence in machines that can learn, reason, and make decisions."

    elif "who created python" in user_input:
        return "Python was created by Guido van Rossum and released in 1991."

    elif user_input in ["hello", "hi", "hey"]:
        return "Hello! Nice to meet you. How can I help you?"

    elif "thank you" in user_input:
        return "You're welcome! Feel free to ask more questions."

    return None

In [5]:
def chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    chat_history_ids = None

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day!")
            break

        # ✅ Step 1: Rule-based answer
        rule_response = rule_based_response(user_input)
        if rule_response:
            print("Chatbot:", rule_response)
            continue

        # ✅ Step 2: Model response
        new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        if chat_history_ids is not None:
            input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        else:
            input_ids = new_input_ids

        chat_history_ids = model.generate(
            input_ids,
            max_length=500,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.7
        )

        response = tokenizer.decode(
            chat_history_ids[:, input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        # ✅ Step 3: Clean response
        response = clean_response(response)

        print("Chatbot:", response)

input_text = "What is Artificial Intelligence?"
input_ids = tokenizer.encode(input_text + tokenizer.eos_token, return_tensors='pt')

output = model.generate(input_ids, max_length=100)
response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)

print("Chatbot:", response)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: It's a new game from the makers of the original Star Wars.


In [6]:
chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: hello
Chatbot: Hello! Nice to meet you. How can I help you?
User: what is ai
Chatbot: Artificial Intelligence is the simulation of human intelligence in machines that can learn, reason, and make decisions.
User: thank you
Chatbot: You're welcome! Feel free to ask more questions.
User: exit
Chatbot: Goodbye! Have a great day!
